# Phase 12: GBM stack — turn every signal into features (the 11 -> 7.6 leap)

Reading the public 7.6 solution settled it: their architecture **is** ours
(PF on the TVT+Z offset, beam search, hold, per-well selector). The difference is
a **gradient-boosting stack** that consumes our every "failed" method as
*features* rather than picking one as the answer:
- PF prediction + per-well PF/GR quality scalars,
- multi-scale GR-template matching (NCC) — the matcher we falsified, now a feature,
- **formation-plane TVT** from neighbour wells (our notebook-11 surface, but as a
  feature the tree can trust *per row* instead of a standalone predictor — which is
  exactly why theirs works on the non-overlap wells where ours failed),
- inter-signal agreement (std/mean across all signals),
- trajectory slopes, GR rolling stats, affine GR calibration.

Target is the **residual off the anchor** (`TVT − last_known_TVT`) so the trees
stay in a bounded range (extrapolation-safe). Models: LightGBM ×3 + CatBoost ×2,
out-of-fold by **well** (GroupKFold), blended by a positive-weight Ridge.

Decision metric: OOF pooled/per-well RMSE vs the PF's ~10.8/12.8 and the public
~7.6. Credit: feature design from `test-fle3n-rogii-v5` (fle3n lineage).

## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

RAW_TRAIN = Path("../data/raw/train")      # has TVT + formation columns
CLEAN_DIR = Path("../data/interim/clean")  # cleaned GR/typewell
EXPORT_DIR = Path("../submissions/gbm"); EXPORT_DIR.mkdir(parents=True, exist_ok=True)

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10
N_FOLDS = 5
REAL_EVAL_FRAC = 0.73

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
def tail_mask(n, frac):
    k = int(round(n * frac)); m = np.zeros(n, bool)
    if k: m[n - k:] = True
    return m

try:
    from lightgbm import LGBMRegressor; HAVE_LGB = True
except Exception: HAVE_LGB = False
try:
    from catboost import CatBoostRegressor; HAVE_CB = True
except Exception: HAVE_CB = False
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
print(f"lgb={HAVE_LGB} catboost={HAVE_CB}")

raw_files = sorted(RAW_TRAIN.glob("*__horizontal_well.csv"))
WELLS = [f.name.replace("__horizontal_well.csv", "") for f in raw_files]
print(f"{len(WELLS)} train wells (raw, with formations)")

lgb=True catboost=False
773 train wells (raw, with formations)


## Engines reused (PF) + signal helpers

PF is our validated v1 engine. Beam is the conservative config. Multi-scale NCC
and the formation-plane KNN follow the public solution. All are computed on a
held-out tail mask so the GBM trains on honest signals.

In [2]:
def _prep(hw, tw):
    tw_s = tw.sort_values("TVT")
    twt = tw_s["TVT"].values.astype(float)
    twg = tw_s["GR"].ffill().bfill().values.astype(float)
    return twt, twg

def run_pf(twt, twg, tvt, Z, MD, gr, kn, ev, N=400, spread=4.5, MOM=0.998,
           VN=0.002, PN=0.005, seed=42):
    last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt[kn], twt, twg)), 10.0, 60.0))
    tl = kn[-30:]; dt = np.diff(tvt[tl]); dz = np.diff(Z[tl]); dm = np.diff(MD[tl]); ok = dm > 0
    ir = float(np.median((dt + dz)[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
    rng = np.random.default_rng(seed)
    pos = (tvt[last] + Z[last]) + spread * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N); w = np.ones(N) / N
    out = np.empty(len(ev)); prev = MD[last]; lo, hi = twt[0] - 100, twt[-1] + 100
    for i, idx in enumerate(ev):
        dmS = max(MD[idx] - prev, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dmS + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - Z[idx], lo, hi); pos = tvt_p + Z[idx]
        g = gr[idx]
        if np.isfinite(g):
            d2 = ((g - np.interp(tvt_p, twt, twg)) / gs) ** 2
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d2, 600.0)), 1e-300)
            s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        if 1.0 / np.sum(w * w) < 0.5 * N:
            ci = np.clip(np.searchsorted(np.cumsum(w), (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1)
            pos = pos[ci] + 0.1 * rng.standard_normal(N); rate = rate[ci] + 0.001 * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = np.sum(w * (pos - Z[idx])); prev = MD[idx]
    return out

def run_beam(twt, twg, tvt, gr, kn, ev, BS=10, mc=20.0, es=144.0, smooth=2):
    g = pd.Series(gr).rolling(smooth, min_periods=1, center=True).mean().values if smooth > 1 else gr
    si = int(np.argmin(np.abs(twt - tvt[kn[-1]]))); beams = {si: 0.0}; hist = []
    for i in ev:
        gv = g[i]; cand = {}
        for idx, cost in beams.items():
            for d in (-2, -1, 0, 1, 2):
                ni = idx + d
                if ni < 0 or ni >= len(twt): continue
                tot = cost + (gv - twg[ni]) ** 2 / es + mc * abs(d)
                if ni not in cand or tot < cand[ni][0]: cand[ni] = (tot, idx)
        top = sorted(cand.items(), key=lambda kv: kv[1][0])[:BS]
        hist.append({ni: p for ni, (c, p) in top}); beams = {ni: c for ni, (c, p) in top}
    best = min(beams, key=beams.get); path = [best]
    for hm in reversed(hist[1:]): best = hm.get(best, best); path.append(best)
    return twt[np.array(path[::-1])]

def multiscale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for h in hws:
        win = 2 * h + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values
        sts = np.arange(0, nk - win + 1, stride)
        if len(sts) == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win)[None, :]]
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, h, mode="edge")
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]]
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win; best = ncc.argmax(1); score = ncc.max(1)
        out.append((ktvt[np.clip(sts[best] + h, 0, nk - 1)].astype(np.float32), score.astype(np.float32)))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3.0 * scores); sw /= sw.sum(1, keepdims=True) + 1e-9
    return out, (tvts * sw).sum(1).astype(np.float32)

class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            try:
                df = pd.read_csv(data_dir / f"{wid}__horizontal_well.csv",
                                 usecols=["X", "Y"] + FORMATIONS).dropna()
            except Exception: continue
            if len(df) == 0: continue
            row = {"wid": wid, "x": float(df["X"].median()), "y": float(df["Y"].median())}
            for c in FORMATIONS: row[f"{c}_m"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df["wid"])}
        xy = self.df[["x", "y"]].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1.0, xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df["x"].to_numpy(); self.ya = self.df["y"].to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf)
        dist = np.atleast_2d(dist); idx = np.atleast_2d(idx)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1); ik = np.take_along_axis(idx, ordr, 1)
        w = np.where(np.isfinite(dk), 1.0 / (dk + 1e-3), 0.0)
        # weighted-mean plane value per formation (robust, fast; not full LS plane)
        fn = self.fa[ik]                              # (npts, k, nForm)
        wsum = w.sum(1, keepdims=True) + 1e-9
        vals = (fn * w[:, :, None]).sum(1) / wsum     # (npts, nForm)
        return vals.astype(np.float32), dk.min(1).astype(np.float32)

## Build the feature matrix (per well, on the 73% tail)

Target = `TVT − last_known_TVT`. Features = all signals + quality scalars,
broadcast per eval row. Formation/dense planes computed with self exclusion.
Resumable cache to `feat_v12.parquet`.

In [3]:
FI = FormationPlaneKNN(WELLS, RAW_TRAIN)
print(f"FormationPlaneKNN built on {len(FI.df)} wells")

def build_well(wid, is_train=True):
    try:
        hw = pd.read_csv(RAW_TRAIN / f"{wid}__horizontal_well.csv").sort_values("MD").reset_index(drop=True)
        twf = CLEAN_DIR / "train" / f"{wid}__typewell.csv"
        tw = pd.read_csv(twf) if twf.exists() else pd.read_csv(RAW_TRAIN / f"{wid}__typewell.csv")
    except Exception:
        return None
    n = len(hw)
    if "TVT" not in hw or hw["TVT"].isna().all() or n < 40: return None
    tvt = hw["TVT"].values.astype(float); Z = hw["Z"].values.astype(float)
    MD = hw["MD"].values.astype(float)
    gr = pd.Series(hw["GR"].values).interpolate(limit_direction="both").bfill().ffill().values.astype(float)
    m = tail_mask(n, REAL_EVAL_FRAC); kn = np.where(~m)[0]; ev = np.where(m)[0]
    if len(kn) < 10 or len(ev) < 5: return None
    twt, twg = _prep(hw, tw)
    if len(twt) < 3: return None
    last_tvt = float(tvt[kn[-1]]); last_z = float(Z[kn[-1]])
    ktvt = tvt[kn]; kz = Z[kn]; kgr = gr[kn]; kmd = MD[kn]

    pf = run_pf(twt, twg, tvt, Z, MD, gr, kn, ev)
    beam = run_beam(twt, twg, tvt, gr, kn, ev)
    sc_res, sc_ens = multiscale_ncc(kgr.astype(np.float32), ktvt.astype(np.float32), gr[ev].astype(np.float32))
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]

    # quality scalars (known-zone)
    tw_at_k = np.interp(ktvt, twt, twg)
    pfx_rmse = rmse(kgr, tw_at_k)
    def rslope(x, y):
        mm = np.isfinite(x) & np.isfinite(y)
        return float(np.polyfit(x[mm], y[mm], 1)[0]) if mm.sum() > 2 and np.std(x[mm]) > 1e-6 else 0.0
    slp_all = rslope(kmd, ktvt); slp50 = rslope(kmd[-50:], ktvt[-50:]); slp_z = rslope(kz, ktvt)
    a_cal, b_cal = (np.polyfit(tw_at_k, kgr, 1) if np.std(tw_at_k) > 1e-6 else (1.0, 0.0))

    # formation-plane TVTs (self-excluded)
    swid = wid if is_train else None
    form_ev, knn_d = FI.impute(hw.loc[ev, ["X", "Y"]].to_numpy(np.float64), self_wid=swid)
    form_kn, _ = FI.impute(hw.loc[kn, ["X", "Y"]].to_numpy(np.float64), self_wid=swid)
    z_ev = Z[ev]
    form_tvts = {}
    form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_well = float(np.median(ktvt + kz - form_kn[:, fi2]))
        tvt_f = (-z_ev + form_ev[:, fi2] + b_well).astype(np.float32)
        form_tvts[f"tvtF_{fn}"] = tvt_f
        form_tvts[f"bw_{fn}"] = np.float32(b_well)
        form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean = fs.mean(1).astype(np.float32); form_std = fs.std(1).astype(np.float32)

    # inter-signal agreement
    sig_mat = np.stack([pf, beam, sc8, sc15, sc25, sc_ens, form_tvts["tvtF_ANCC"], form_mean], 1)
    sig_std = sig_mat.std(1).astype(np.float32); sig_mean = sig_mat.mean(1).astype(np.float32)

    # GR rolling at eval rows
    grs = pd.Series(gr)
    roll = {f"gr_roll{w}": grs.rolling(w, center=True, min_periods=1).mean().values[ev].astype(np.float32)
            for w in (5, 21, 51)}

    md_ev = MD[ev]
    feat = {
        "well": wid, "row_idx": ev.astype(np.int32),
        "target": (tvt[ev] - last_tvt).astype(np.float32),
        # signals as residual-from-anchor (bounded)
        "pf": (pf - last_tvt).astype(np.float32),
        "beam": (beam - last_tvt).astype(np.float32),
        "sc8": (sc8 - last_tvt), "sc15": (sc15 - last_tvt), "sc25": (sc25 - last_tvt),
        "sc_ens": (sc_ens - last_tvt),
        "sc8s": sc8s, "sc15s": sc15s, "sc25s": sc25s,
        "form_mean": (form_mean - last_tvt), "form_std": form_std,
        "sig_std": sig_std, "sig_mean": (sig_mean - last_tvt),
        "d_md": (md_ev - MD[kn[-1]]).astype(np.float32),
        "d_z": (z_ev - last_z).astype(np.float32),
        "gr_ev": gr[ev].astype(np.float32),
        "knn_d": knn_d,
        # broadcast scalars
        "pfx_rmse": np.float32(pfx_rmse), "slp_all": np.float32(slp_all),
        "slp50": np.float32(slp50), "slp_z": np.float32(slp_z),
        "a_cal": np.float32(a_cal), "b_cal": np.float32(b_cal),
        "n_known": np.int32(len(kn)), "n_eval": np.int32(len(ev)),
        "z_span": np.float32(Z.max() - Z.min()),
    }
    feat.update({k: (v - last_tvt).astype(np.float32) if k.startswith("tvtF_") else v
                 for k, v in form_tvts.items()})
    feat.update(roll)
    df = pd.DataFrame({k: (np.full(len(ev), v) if np.isscalar(v) or (hasattr(v, "ndim") and v.ndim == 0) else v)
                       for k, v in feat.items()})
    return df

CACHE = Path("../data/interim/feat_v12.parquet")
if CACHE.exists():
    feat_all = pd.read_parquet(CACHE)
    print(f"loaded cached features: {len(feat_all):,} rows, {feat_all['well'].nunique()} wells")
else:
    t0 = time.time(); parts = []
    for i, wid in enumerate(WELLS):
        d = build_well(wid, is_train=True)
        if d is not None: parts.append(d)
        if (i + 1) % 50 == 0:
            print(f"  ...{i+1}/{len(WELLS)} wells [{(time.time()-t0)/60:.1f} min]")
    feat_all = pd.concat(parts, ignore_index=True)
    feat_all.to_parquet(CACHE)
    print(f"built features: {len(feat_all):,} rows, {feat_all['well'].nunique()} wells "
          f"[{(time.time()-t0)/60:.1f} min]")

FormationPlaneKNN built on 765 wells
loaded cached features: 3,717,363 rows, 773 wells


## Train the GBM stack out-of-fold by well + Ridge blend

In [4]:
FEATS = [c for c in feat_all.columns if c not in ("well", "row_idx", "target")]
print(f"{len(FEATS)} features")
X = feat_all[FEATS].values.astype(np.float32)
y = feat_all["target"].values.astype(np.float32)
wells_arr = feat_all["well"].values
uniq = np.array(sorted(feat_all["well"].unique()))
rng = np.random.default_rng(0)
fold_of = {w: i for i, w in zip(rng.permutation(len(uniq)) % N_FOLDS, uniq)}
folds = np.array([fold_of[w] for w in wells_arr])

def make_models():
    ms = []
    if HAVE_LGB:
        ms += [("lgb1", LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=63,
                                      subsample=0.8, colsample_bytree=0.8, verbose=-1)),
               ("lgb2", LGBMRegressor(n_estimators=800, learning_rate=0.02, num_leaves=31,
                                      subsample=0.7, colsample_bytree=0.7, verbose=-1))]
    if HAVE_CB:
        ms += [("cb1", CatBoostRegressor(iterations=600, learning_rate=0.03, depth=8,
                                         verbose=0, allow_writing_files=False))]
    if not ms:
        ms = [("hgb1", HistGradientBoostingRegressor(max_iter=600, learning_rate=0.03)),
              ("hgb2", HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_depth=6))]
    return ms

base = make_models()
oof = {name: np.zeros(len(y)) for name, _ in base}
t0 = time.time()
for f in range(N_FOLDS):
    tr = folds != f; va = folds == f
    for name, mk in base:
        mdl = mk.__class__(**mk.get_params())
        mdl.fit(X[tr], y[tr])
        oof[name][va] = mdl.predict(X[va])
    print(f"  fold {f} done [{(time.time()-t0)/60:.1f} min]")

# anchor for reconstruction: target is residual; abs TVT = residual + last_tvt.
# we evaluate residual RMSE directly (== TVT RMSE since anchor is a per-row constant).
def pooled(resid_pred): return rmse(resid_pred, y)
print("\nbase model OOF residual RMSE (== TVT RMSE):")
for name in oof: print(f"  {name:6s} {pooled(oof[name]):.3f}")

# Ridge blend (positive weights via clipped Ridge on OOF)
Z = np.column_stack([oof[n] for n in oof])
ridge = Ridge(alpha=1.0, positive=True).fit(Z, y)
blend = ridge.predict(Z)
print(f"\nRidge blend OOF RMSE: {pooled(blend):.3f}")
print("blend weights:", {n: round(float(w),3) for n,w in zip(oof, ridge.coef_)})

41 features
  fold 0 done [1.0 min]
  fold 1 done [2.1 min]
  fold 2 done [3.1 min]
  fold 3 done [4.2 min]
  fold 4 done [5.2 min]

base model OOF residual RMSE (== TVT RMSE):
  lgb1   13.800
  lgb2   13.723

Ridge blend OOF RMSE: 13.701
blend weights: {'lgb1': 0.315, 'lgb2': 0.689}


In [5]:
# per-well pooled + per-well-mean, and the comparison that decides everything
err = blend - y
per_well = feat_all.assign(err=err).groupby("well")["err"].apply(lambda e: np.sqrt(np.mean(e**2)))
# floor on the same rows (residual target's floor = predicting 0 == hold-last)
floor_resid = rmse(np.zeros_like(y), y)
print(f"{'':14s}{'pooled':>9s}{'per-well':>10s}")
print(f"{'floor(hold)':14s}{floor_resid:9.3f}{feat_all.assign(e=y).groupby('well')['e'].apply(lambda e: np.sqrt(np.mean(e**2))).mean():10.3f}")
print(f"{'GBM blend':14s}{pooled(blend):9.3f}{per_well.mean():10.3f}")
print(f"\nreference: PF v4 ~12.8 pooled / 10.8 per-well | public stack ~7.6")
print("If blend pooled << 12.8, the stack is the leap; promote to submission.")

import joblib
joblib.dump({"feats": FEATS, "base": base, "ridge_coef": ridge.coef_.tolist(),
             "oof_pooled": pooled(blend)}, EXPORT_DIR / "gbm_stack_meta.pkl")
print("saved gbm_stack_meta.pkl")

                 pooled  per-well
floor(hold)      16.371    13.423
GBM blend        13.701    11.270

reference: PF v4 ~12.8 pooled / 10.8 per-well | public stack ~7.6
If blend pooled << 12.8, the stack is the leap; promote to submission.
saved gbm_stack_meta.pkl


## Reading & next

- **base model RMSE**: each booster alone vs the PF (~12.8). Trees should already
  beat the PF because they see all signals + quality scalars at once.
- **Ridge blend**: the headline. If it lands near the public ~7.6, the diagnosis
  was right and this is the model. The blend weights show which boosters matter.
- The **formation-plane features** (`tvtF_*`, `form_mean`) are the new signal that
  failed as a standalone predictor (nb11) but should earn weight here — check LGBM
  feature importance to confirm.
- To submit: refit all base models on the full data (no held-out fold), and in
  `submission.ipynb` build the same feature matrix for the test wells (plane KNN
  uses the *train* wells, no self-exclusion needed since test wells aren't in it),
  predict residual, add the anchor. The overlap exploit still applies first.